In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [ ]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle, according to the most recent publications?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

In [ ]:
"""

EXAMPLE OF CLAUDE RESPONSE WHEN USING WEB SEARCH TOOL

[
    # Exact query Claude executed
    ServerToolUseBlock(
        id='srvtoolu_01B6iwpcgerLKQMhFVuTNdf2', 
        input={'query': 'best exercise leg muscle gain recent research 2026'}, 
        name='web_search', 
        type='server_tool_use', 
        caller={'type': 'direct'}
    ), 
    
    # Search results that Claude found
    WebSearchToolResultBlock(
        content=[
            WebSearchResultBlock(
                encrypted_content='EogfC...', 
                page_age=None, 
                title='Loading Recommendations for Muscle Strength, Hypertrophy, and Local Endurance: A Re-Examination of the Repetition Continuum - PMC', 
                type='web_search_result', 
                url='https://pmc.ncbi.nlm.nih.gov/articles/PMC7927075/'
            ),
            # ...More search results...
        ],
    ),

    # Now Claude begin to answer the user's question alternating TextBlocks with or without Citations

    # Answering without a Citation
    TextBlock(
        citations=None, 
        text='Based on the recent research I\'ve found, ...', 
        type='text'
    ),

    # Answering with a Citation
    TextBlock(
        citations=[
            CitationsWebSearchResultLocation(
                cited_text='It is essential to emphasize that squat and leg press exercises are crucial for promoting hypertrophic gains across various muscle groups, particularl...', 
                encrypted_index='EpABCioID...', 
                title='The impact of resistance training on gluteus maximus hypertrophy: a systematic review and meta-analysis - PMC', 
                type='web_search_result_location', 
                url='https://pmc.ncbi.nlm.nih.gov/articles/PMC12018462/'
            )
        ], 
        text='Squat and leg press exercises are crucial for promoting hypertrophic gains across various muscle groups, particularly the quadriceps femoris', 
        type='text'
    ), 

    # ...More TextBlocks (with or without citations)...
]

"""